# 06 · Zarr IO and Mtb Readout QC
Quick-look notebook for inspecting a single zarr position: loads images,
segmentation, and Trackastra track table, then launches a combined napari view.
Use this to verify that the full pipeline output is consistent before analysis.

In [ ]:
import napari
import zarr
import dask.array as da
import pandas as pd
import numpy as np
from pathlib import Path

## 1 · Configure position

In [ ]:
zarr_path = Path('/mnt/DATA3/BPP0050/BPP0050-1-Live-cell-to4i_Live-1__2025-04-09T18_25_04-Measurement 1/live_1.zarr')
pos = '3_3'
N_PREVIEW = 48  # number of frames to preview

## 2 · Load data

In [ ]:
zarr_root = zarr.open_group(str(zarr_path / pos), mode='r')

image_array = zarr_root['images']['0']
seg_array   = zarr_root['labels']['trackastra_labels']['0']

image_preview = image_array.images[:N_PREVIEW].max(axis=2)  # Z-max: (T, C, Y, X)
seg_preview   = seg_array[:N_PREVIEW]                        # (T, Y, X)

print('Image shape:', image_preview.shape)
print('Segmentation shape:', seg_preview.shape)

tracks_path = zarr_path / pos / 'labels' / 'trackastra_labels' / 'tables' / 'track'
cols = [f.name for f in tracks_path.iterdir() if f.is_dir()]
track_df = pd.DataFrame({col: zarr.open(tracks_path / col, mode='r')[:] for col in cols})
print(f'Track table: {len(track_df)} rows, {track_df.track_id.nunique()} unique tracks')
track_df.head()

## 3 · Napari visualisation

In [ ]:
track_subset = track_df[track_df['t'] < N_PREVIEW]
coords = track_subset[['track_id', 't', 'y', 'x']].values

viewer = napari.Viewer()
viewer.add_image(image_preview, channel_axis=1, name='Live images')
viewer.add_labels(seg_preview, name='Segmentation')
viewer.add_tracks(coords, name='Tracks')
napari.run()